# RSSM Weekly Evaluation Preparation

This notebook downloads the trained RSSM/world-model checkpoint from Hugging Face and checks that the artifact is compatible with the local evaluation code. It inspects the config, normalization stats, embedding files, and checkpoint structure, then attempts a minimal local model load and smoke test. It intentionally does not run full evaluation.

## Setup

In [1]:
from __future__ import annotations

from pathlib import Path
import importlib.util
import json
import subprocess
import sys

import numpy as np
import pandas as pd
import torch
import yaml

if importlib.util.find_spec("huggingface_hub") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "huggingface_hub"])

from huggingface_hub import snapshot_download

def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / "model").is_dir() and (candidate / "eval").is_dir() and (candidate / "configs").is_dir():
            return candidate
    raise RuntimeError(f"Could not find project root from {start}")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"torch = {torch.__version__}")

PROJECT_ROOT = C:\stocktwits_2026\StockTwit_WM
torch = 2.12.0.dev20260408+cu128


In [2]:
MODEL_REPO_ID = "abhi0710/twitwave-rssm-large"
MODEL_DIR = PROJECT_ROOT / "external_models" / "twitwave-rssm-large"
CHECKPOINT_PATH = MODEL_DIR / "checkpoints" / "best.pt"
CONFIG_PATH = MODEL_DIR / "config.yaml"
NORM_STATS_PATH = MODEL_DIR / "norm_stats.json"
EMBEDDINGS_DIR = MODEL_DIR / "embeddings"

ALLOW_PATTERNS = [
    "config.yaml",
    "norm_stats.json",
    "checkpoints/best.pt",
    "embeddings/*",
    "logs/kl_log.json",
]

MODEL_DIR.mkdir(parents=True, exist_ok=True)
print(f"MODEL_DIR = {MODEL_DIR}")

MODEL_DIR = C:\stocktwits_2026\StockTwit_WM\external_models\twitwave-rssm-large


## Download Selected Model Files

In [3]:
downloaded_path = snapshot_download(
    repo_id=MODEL_REPO_ID,
    repo_type="model",
    local_dir=MODEL_DIR,
    allow_patterns=ALLOW_PATTERNS,
)

print(f"Downloaded or reused files under: {downloaded_path}")

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Downloaded or reused files under: C:\stocktwits_2026\StockTwit_WM\external_models\twitwave-rssm-large


## Required File Checklist

In [4]:
required_files = {
    "config.yaml": CONFIG_PATH,
    "norm_stats.json": NORM_STATS_PATH,
    "checkpoints/best.pt": CHECKPOINT_PATH,
    "embeddings/ticker_to_idx.json": EMBEDDINGS_DIR / "ticker_to_idx.json",
    "embeddings/idx_to_ticker.json": EMBEDDINGS_DIR / "idx_to_ticker.json",
    "embeddings/e_dec.npy": EMBEDDINGS_DIR / "e_dec.npy",
    "embeddings/e_ret.npy": EMBEDDINGS_DIR / "e_ret.npy",
}

missing = []
for label, path in required_files.items():
    exists = path.exists()
    status = "OK" if exists else "MISSING"
    print(f"[{status:7}] {label} -> {path}")
    if not exists:
        missing.append(label)

if missing:
    raise FileNotFoundError("Missing required model files: " + ", ".join(missing))

[OK     ] config.yaml -> C:\stocktwits_2026\StockTwit_WM\external_models\twitwave-rssm-large\config.yaml
[OK     ] norm_stats.json -> C:\stocktwits_2026\StockTwit_WM\external_models\twitwave-rssm-large\norm_stats.json
[OK     ] checkpoints/best.pt -> C:\stocktwits_2026\StockTwit_WM\external_models\twitwave-rssm-large\checkpoints\best.pt
[OK     ] embeddings/ticker_to_idx.json -> C:\stocktwits_2026\StockTwit_WM\external_models\twitwave-rssm-large\embeddings\ticker_to_idx.json
[OK     ] embeddings/idx_to_ticker.json -> C:\stocktwits_2026\StockTwit_WM\external_models\twitwave-rssm-large\embeddings\idx_to_ticker.json
[OK     ] embeddings/e_dec.npy -> C:\stocktwits_2026\StockTwit_WM\external_models\twitwave-rssm-large\embeddings\e_dec.npy
[OK     ] embeddings/e_ret.npy -> C:\stocktwits_2026\StockTwit_WM\external_models\twitwave-rssm-large\embeddings\e_ret.npy


## Config Inspection

In [5]:
with CONFIG_PATH.open("r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

def nested_get(d: dict, *keys, default=None):
    cur = d
    for key in keys:
        if not isinstance(cur, dict) or key not in cur:
            return default
        cur = cur[key]
    return cur

model_cfg = cfg.get("model", {}) if isinstance(cfg, dict) else {}
train_cfg = cfg.get("train", {}) if isinstance(cfg, dict) else {}
eval_cfg = cfg.get("eval", {}) if isinstance(cfg, dict) else {}

summary_items = {
    "top_k": model_cfg.get("top_k"),
    "feature_dim": model_cfg.get("feature_dim"),
    "seq_len": train_cfg.get("seq_len"),
    "context_len": eval_cfg.get("context_len"),
    "horizons": eval_cfg.get("horizons"),
    "h_dim": model_cfg.get("h_dim"),
    "s_dim": model_cfg.get("s_dim"),
    "d_enc": model_cfg.get("d_enc"),
    "embed_dim": model_cfg.get("embed_dim"),
    "window_k": model_cfg.get("window_k"),
    "n_layers": model_cfg.get("n_layers"),
    "n_heads": model_cfg.get("n_heads"),
    "dropout": model_cfg.get("dropout"),
    "feature_columns": cfg.get("feature_columns") or cfg.get("features") or nested_get(cfg, "data", "feature_columns"),
}

print("Top-level config keys:", list(cfg.keys()))
for key, value in summary_items.items():
    print(f"{key:16}: {value}")

Top-level config keys: ['eval', 'model', 'train']
top_k           : 100
feature_dim     : 5
seq_len         : 64
context_len     : 52
horizons        : [1, 4, 13]
h_dim           : 1024
s_dim           : 256
d_enc           : 512
embed_dim       : 128
window_k        : 8
n_layers        : 4
n_heads         : 16
dropout         : 0.15
feature_columns : None


## Normalization Stats Inspection

In [6]:
with NORM_STATS_PATH.open("r", encoding="utf-8") as f:
    norm_stats = json.load(f)

print("norm_stats keys:", list(norm_stats.keys()))

mean = np.asarray(norm_stats.get("mean", []), dtype=float)
std = np.asarray(norm_stats.get("std", []), dtype=float)
print(f"mean shape: {mean.shape}, std shape: {std.shape}")

feature_dim = model_cfg.get("feature_dim")
if feature_dim is not None:
    mean_ok = mean.size == int(feature_dim)
    std_ok = std.size == int(feature_dim)
    print(f"mean length matches feature_dim={feature_dim}: {mean_ok}")
    print(f"std length matches feature_dim={feature_dim}: {std_ok}")
    if not (mean_ok and std_ok):
        raise ValueError("Normalization mean/std lengths do not match config model.feature_dim")

norm_stats keys: ['mean', 'std']
mean shape: (5,), std shape: (5,)
mean length matches feature_dim=5: True
std length matches feature_dim=5: True


## Embeddings Inspection

In [7]:
e_dec = np.load(EMBEDDINGS_DIR / "e_dec.npy")
e_ret = np.load(EMBEDDINGS_DIR / "e_ret.npy")

with (EMBEDDINGS_DIR / "ticker_to_idx.json").open("r", encoding="utf-8") as f:
    ticker_to_idx = json.load(f)
with (EMBEDDINGS_DIR / "idx_to_ticker.json").open("r", encoding="utf-8") as f:
    idx_to_ticker = json.load(f)

print(f"e_dec shape: {e_dec.shape}")
print(f"e_ret shape: {e_ret.shape}")
print(f"ticker_to_idx entries: {len(ticker_to_idx)}")
print(f"idx_to_ticker entries: {len(idx_to_ticker)}")

print("First 10 ticker mappings:")
for ticker, idx in list(ticker_to_idx.items())[:10]:
    print(f"  {ticker}: {idx}")

top_k = model_cfg.get("top_k")
if top_k is not None:
    print(f"ticker count >= top_k={top_k}: {len(ticker_to_idx) >= int(top_k)}")

if e_dec.shape != e_ret.shape:
    raise ValueError("e_dec and e_ret shapes differ")

e_dec shape: (1521, 128)
e_ret shape: (1521, 128)
ticker_to_idx entries: 1520
idx_to_ticker entries: 1521
First 10 ticker mappings:
  6E_F: 1
  AA: 2
  AAL: 3
  AAOI: 4
  AAP: 5
  AAPL: 6
  ABAT: 7
  ABBV: 8
  ABIL: 9
  ABIO: 10
ticker count >= top_k=100: True


## Checkpoint Inspection

PyTorch `.pt` checkpoints are pickle-based. Only load this file if you trust the source of the model artifact.

In [8]:
ckpt = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
print("checkpoint type:", type(ckpt))

def tensor_shape(value):
    return tuple(value.shape) if torch.is_tensor(value) else None

def find_state_dict(checkpoint):
    if isinstance(checkpoint, dict):
        for key in ("model_state", "model_state_dict", "state_dict", "model"):
            value = checkpoint.get(key)
            if isinstance(value, dict) and any(torch.is_tensor(v) for v in value.values()):
                return key, value
        if checkpoint and all(torch.is_tensor(v) for v in checkpoint.values()):
            return "<raw_state_dict>", checkpoint
    return None, None

if isinstance(ckpt, dict):
    print("checkpoint keys:", list(ckpt.keys()))
    for key in ("epoch", "global_step", "best_val_loss", "metrics", "model_cfg", "train_cfg", "config", "optimizer_state", "optimizer_state_dict"):
        if key in ckpt:
            value = ckpt[key]
            if isinstance(value, dict):
                print(f"{key}: dict with {len(value)} keys")
                if key == "model_cfg":
                    print("  " + json.dumps(value, sort_keys=True))
            else:
                print(f"{key}: {value}")

state_key, state_dict = find_state_dict(ckpt)
print("state_dict key:", state_key)
if state_dict is not None:
    print(f"state_dict parameters: {len(state_dict)}")
    for name, value in list(state_dict.items())[:12]:
        print(f"  {name}: {tensor_shape(value)}")
else:
    print("No obvious model state_dict found in checkpoint")

checkpoint type: <class 'dict'>
checkpoint keys: ['model_state', 'optimizer_state', 'lr_sched_state', 'beta_sched_state', 'global_step', 'epoch', 'best_val_loss', 'best_val_recon', 'epochs_no_improve', 'model_cfg', 'train_cfg', 'kl_log']
epoch: 33
global_step: 759
best_val_loss: 2.2642432848612466
model_cfg: dict with 12 keys
  {"d_enc": 512, "dropout": 0.15, "embed_dim": 128, "feature_dim": 5, "h_dim": 1024, "mlp_hidden": 512, "n_heads": 16, "n_layers": 4, "s_dim": 256, "top_k": 100, "vocab_size": 1521, "window_k": 8}
train_cfg: dict with 17 keys
optimizer_state: dict with 2 keys
state_dict key: model_state
state_dict parameters: 111
  embeddings.e_ret.weight: (1521, 128)
  embeddings.e_dec.weight: (1521, 128)
  set_encoder.input_proj.weight: (512, 133)
  set_encoder.input_proj.bias: (512,)
  set_encoder.transformer.layers.0.self_attn.in_proj_weight: (1536, 512)
  set_encoder.transformer.layers.0.self_attn.in_proj_bias: (1536,)
  set_encoder.transformer.layers.0.self_attn.out_proj.wei

## Compatibility Check With Local Repo Code

In [9]:
try:
    from model.twit_wave import ModelConfig, TwitWave
    print("Imported TwitWave and ModelConfig from local repo")
except Exception as exc:
    raise RuntimeError("Could not import local TwitWave model code. Consider refactoring a shared loader into eval/utils.py.") from exc

def clean_state_dict(sd: dict) -> dict:
    cleaned = {}
    for key, value in sd.items():
        new_key = key[len("module."):] if key.startswith("module.") else key
        cleaned[new_key] = value
    return cleaned

vocab_size = int(e_dec.shape[0])
compat_cfg = ModelConfig(
    vocab_size=vocab_size,
    embed_dim=int(model_cfg.get("embed_dim", e_dec.shape[1])),
    d_enc=int(model_cfg.get("d_enc", 256)),
    h_dim=int(model_cfg.get("h_dim", 256)),
    s_dim=int(model_cfg.get("s_dim", 128)),
    n_heads=int(model_cfg.get("n_heads", 4)),
    n_layers=int(model_cfg.get("n_layers", 2)),
    window_k=int(model_cfg.get("window_k", 4)),
    mlp_hidden=int(model_cfg.get("mlp_hidden", 256)),
    feature_dim=int(model_cfg.get("feature_dim", 5)),
    top_k=int(model_cfg.get("top_k", 100)),
    dropout=0.0,
)

model = TwitWave(compat_cfg)
local_state = model.state_dict()
cleaned_state = clean_state_dict(state_dict) if state_dict is not None else None
shape_mismatches = []
if cleaned_state is not None:
    for key, value in cleaned_state.items():
        if key in local_state and torch.is_tensor(value) and tuple(value.shape) != tuple(local_state[key].shape):
            shape_mismatches.append((key, tuple(value.shape), tuple(local_state[key].shape)))

load_result = None
load_error = None

if state_dict is None:
    load_error = "No state_dict was found in checkpoint"
elif shape_mismatches:
    load_error = f"Found {len(shape_mismatches)} parameter shape mismatch(es)"
else:
    try:
        load_result = model.load_state_dict(cleaned_state, strict=False)
        model.eval()
    except Exception as exc:
        load_error = repr(exc)

print("ModelConfig used for compatibility check:")
print(compat_cfg)
if shape_mismatches:
    print("Parameter shape mismatches against current repo model code:")
    for key, ckpt_shape, local_shape in shape_mismatches[:20]:
        print(f"  {key}: checkpoint={ckpt_shape}, local={local_shape}")
    if len(shape_mismatches) > 20:
        print(f"  ... {len(shape_mismatches) - 20} more")

if load_result is not None:
    print("Checkpoint load completed with strict=False")
    print("missing keys:", list(load_result.missing_keys)[:20], "count=", len(load_result.missing_keys))
    print("unexpected keys:", list(load_result.unexpected_keys)[:20], "count=", len(load_result.unexpected_keys))
    compatible_load = len(load_result.missing_keys) == 0 and len(load_result.unexpected_keys) == 0
    print("exact key match:", compatible_load)
else:
    compatible_load = False
    print("Checkpoint load failed:", load_error)
    print("TODO: Refactor a shared loader into eval/utils.py once the checkpoint key format is confirmed.")

Imported TwitWave and ModelConfig from local repo


ModelConfig used for compatibility check:
ModelConfig(vocab_size=1521, embed_dim=128, d_enc=512, h_dim=1024, s_dim=256, n_heads=16, n_layers=4, window_k=8, mlp_hidden=512, feature_dim=5, top_k=100, dropout=0.0)
Parameter shape mismatches against current repo model code:
  rssm.gru.weight_ih: checkpoint=(3072, 264), local=(3072, 256)
Checkpoint load failed: Found 1 parameter shape mismatch(es)
TODO: Refactor a shared loader into eval/utils.py once the checkpoint key format is confirmed.


C:\stocktwits_2026\StockTwit_WM\model\set_encoder.py:37: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
C:\stocktwits_2026\StockTwit_WM\model\temporal_encoder.py:49: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)


## Minimal Smoke Test

In [10]:
if not compatible_load:
    print("Skipping smoke test because the checkpoint did not load cleanly.")
else:
    with torch.no_grad():
        batch_size = 1
        h, s = model.rssm.init_state(batch_size, torch.device("cpu"))
        h_new, s_new, z_new, pres_logits = model.forward_step_prior(h, s, use_mean=True)
        n_decode = min(4, compat_cfg.top_k, vocab_size - 1)
        ticker_ids = torch.arange(1, n_decode + 1, dtype=torch.long).unsqueeze(0)
        feat_pred = model.decode_features(z_new, ticker_ids)

    print("forward_step_prior shapes:")
    print("  h_new:", tuple(h_new.shape))
    print("  s_new:", tuple(s_new.shape))
    print("  z_new:", tuple(z_new.shape))
    print("  pres_logits:", tuple(pres_logits.shape))
    print("decode_features shape:", tuple(feat_pred.shape))

Skipping smoke test because the checkpoint did not load cleanly.


## Next Step

If this notebook shows a clean checkpoint load, the next notebook/script should build the weekly evaluation dataset, load `norm_stats.json` into the dataset normalization path, warm up the context window with `model.context_phase`, and run horizon-specific prediction metrics. If loading is not exact, first refactor a shared artifact-aware loader into `eval/utils.py` so notebook and scripts use the same checkpoint/config conventions.